In [6]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils import (
    set_seed,
    prepare_all_fold_tensors,
    run_nested_random_search,
    print_final_results,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

17:07:46 [INFO] Device: cuda


In [7]:
CFG = {
    "task": "hi",
    "dataset": "drd2",
    "fp_type": "rdkit_desc",
    "n_bits": 1024,
    "outer_folds": [1, 2, 3],
    "inner_k": 2,
    "random_state": GLOBAL_SEED,
}

In [8]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],
    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [9]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(
        CFG["task"],
        CFG["dataset"],
        fold_idx,
    )

    folds_data[fold_idx] = {
        "train": train_df,
        "test": test_df,
    }

    n_train = len(train_df)
    n_test = len(test_df)

    n_train_pos = int(train_df["value"].sum())
    n_train_neg = n_train - n_train_pos

    n_test_pos = int(test_df["value"].sum())
    n_test_neg = n_test - n_test_pos

    logger.info(
        f"Fold {fold_idx} | "
        f"train={n_train} "
        f"(pos={n_train_pos}, neg={n_train_neg}, "
        f"pos_ratio={n_train_pos / n_train:.3f}) | "
        f"test={n_test} "
        f"(pos={n_test_pos}, neg={n_test_neg}, "
        f"pos_ratio={n_test_pos / n_test:.3f})"
    )

17:07:46 [INFO] Loaded hi/drd2 fold 1: train=2385, test=1190
17:07:46 [INFO] Fold 1 | train=2385 (pos=1684, neg=701, pos_ratio=0.706) | test=1190 (pos=735, neg=455, pos_ratio=0.618)
17:07:46 [INFO] Loaded hi/drd2 fold 2: train=2381, test=1194
17:07:46 [INFO] Fold 2 | train=2381 (pos=1510, neg=871, pos_ratio=0.634) | test=1194 (pos=909, neg=285, pos_ratio=0.761)
17:07:46 [INFO] Loaded hi/drd2 fold 3: train=2384, test=1191
17:07:46 [INFO] Fold 3 | train=2384 (pos=1644, neg=740, pos_ratio=0.690) | test=1191 (pos=775, neg=416, pos_ratio=0.651)


In [10]:
from utils.mlp_utils import prepare_all_fold_tensors

folds_tensors = prepare_all_fold_tensors(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

17:07:46 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/rdkit_desc_train_1.npz
17:07:46 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/rdkit_desc_test_1.npz
17:07:46 [INFO] Fold 1 | X_train: (2385, 217), X_test: (1190, 217) | scale_features=True | pos_weight=0.416
17:07:46 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/rdkit_desc_train_2.npz
17:07:46 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/rdkit_desc_test_2.npz
17:07:46 [INFO] Fold 2 | X_train: (2381, 217), X_test: (1194, 217) | scale_features=True | pos_weight=0.577
17:07:46 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/rdkit_desc_train_3.npz
17:07:46 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/drd2/rdkit_desc_test_3.npz
17:07:46 [INFO] Fold 3 | X_train: (2384, 21

In [11]:
logger.info(f"Estimated time: ~{N_ITER * 6 * len(CFG['outer_folds']) / 60:.0f} min")

fold_results = run_nested_random_search(
    cfg=CFG,
    folds_tensors=folds_tensors,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

print_final_results(fold_results, title="MLP DRD2 Hi — rdkit_desc")

17:07:46 [INFO] Estimated time: ~45 min
17:07:46 [INFO] 
OUTER FOLD 1
17:07:50 [INFO]   [1/150] inner PR-AUC=0.9306 (4s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
17:07:55 [INFO]   [2/150] inner PR-AUC=0.9150 (5s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
17:07:58 [INFO]   [3/150] inner PR-AUC=0.9215 (3s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
17:08:02 [INFO]   [4/150] inner PR-AUC=0.9199 (4s) | L=2 N=64 r=0.7 dropout=0.3 lr=3e-03
17:08:04 [INFO]   [5/150] inner PR-AUC=0.9242 (2s) | L=2 N=256 r=1.0 dropout=0.5 lr=3e-03
17:08:13 [INFO]   [6/150] inner PR-AUC=0.8932 (9s) | L=1 N=64 r=0.7 dropout=0.5 lr=1e-04
17:08:21 [INFO]   [7/150] inner PR-AUC=0.9225 (8s) | L=3 N=128 r=1.0 dropout=0.3 lr=1e-03
17:08:28 [INFO]   [8/150] inner PR-AUC=0.9002 (7s) | L=2 N=64 r=0.7 dropout=0.5 lr=5e-04
17:08:34 [INFO]   [9/150] inner PR-AUC=0.9118 (7s) | L=1 N=256 r=0.5 dropout=0.3 lr=1e-04
17:08:47 [INFO]   [10/150] inner PR-AUC=0.9107 (12s) | L=3 N=64 r=0.5 dropout=0.0 lr=5e-04
17:09:17 [INFO]   [11/150] inne


MLP DRD2 Hi — rdkit_desc
Fold 1: PR-AUC=0.7191  ROC-AUC=0.6382
Fold 2: PR-AUC=0.8686  ROC-AUC=0.6909
Fold 3: PR-AUC=0.6985  ROC-AUC=0.6201

Aggregated metrics:
  pr_auc_mean: 0.7621
  pr_auc_std: 0.0758
  roc_auc_mean: 0.6497
  roc_auc_std: 0.03
  bedroc_mean: 0.76
  bedroc_std: 0.175
  f1_at_05_mean: 0.7176
  f1_at_05_std: 0.0348
  positive_rate_mean: 0.6765
  positive_rate_std: 0.0614

Best hyperparameters per fold:
Fold 1: L=2 N=256 r=0.5 act=leaky_relu dropout=0.2 bn=False init=xavier lr=1e-03 wd=5e-03 bs=128
Fold 2: L=3 N=256 r=0.7 act=relu dropout=0.3 bn=True init=kaiming lr=2e-03 wd=1e-03 bs=64
Fold 3: L=3 N=1024 r=0.7 act=relu dropout=0.0 bn=True init=xavier lr=1e-03 wd=1e-03 bs=128


{'pr_auc_mean': np.float64(0.7621),
 'pr_auc_std': np.float64(0.0758),
 'roc_auc_mean': np.float64(0.6497),
 'roc_auc_std': np.float64(0.03),
 'bedroc_mean': np.float64(0.76),
 'bedroc_std': np.float64(0.175),
 'f1_at_05_mean': np.float64(0.7176),
 'f1_at_05_std': np.float64(0.0348),
 'positive_rate_mean': np.float64(0.6765),
 'positive_rate_std': np.float64(0.0614)}